# 04 · pandas Time Series

Most data engineering is time-oriented: daily loads, monthly revenue, rolling
averages. pandas has first-class time-series support built on a **datetime
index**. This notebook covers resampling, rolling windows and shifting.

In [ ]:
# ▶ Run this first. Locates the sample data no matter where the kernel starts.
from pathlib import Path

def find_data() -> Path:
    here = Path.cwd()
    for base in (here, *here.parents):
        if (base / 'data' / 'raw').exists():
            return base / 'data'
    raise FileNotFoundError('Run: uv run python data/build_data.py')

DATA = find_data()
RAW = DATA / 'raw'
print('Data directory:', DATA)
print('Raw files:', sorted(p.name for p in RAW.glob('*')))

In [ ]:
import pandas as pd
orders = pd.read_csv(RAW / 'orders.csv', parse_dates=['order_ts'])
orders = orders[orders['status'] == 'completed'].copy()
print('completed orders:', len(orders))
print('date range:', orders['order_ts'].min(), '->', orders['order_ts'].max())

## A datetime index

Setting the timestamp as the index unlocks time-based selection and resampling.
Once indexed by time, you can slice by date strings directly.

In [ ]:
ts = orders.set_index('order_ts').sort_index()
print(ts.loc['2024-03', 'amount'].sum().round(2), 'revenue in March 2024')
print(ts.loc['2024-03-01':'2024-03-07', 'amount'].count(), 'orders in first week of March')

## Resampling: change the time grain

`resample` is `groupby` for time — roll rows up to daily, weekly, monthly
buckets. `'D'`, `'W'`, `'ME'` (month-end) are common rules.

In [ ]:
monthly = ts['amount'].resample('ME').agg(['sum', 'count', 'mean']).round(2)
print('monthly revenue:')
print(monthly.head())

## Rolling windows (moving averages)

A **rolling** window smooths noisy series — e.g. a 7-day moving average of daily
revenue. This is a staple of dashboards and anomaly detection.

In [ ]:
daily = ts['amount'].resample('D').sum()
rolling = daily.rolling(window=7, min_periods=1).mean().round(2)
compare = pd.DataFrame({'daily': daily, 'ma_7d': rolling})
print(compare.head(10))

## Shifting & period-over-period change

`shift` moves values by N periods so you can compare each period to the previous
one — the basis of month-over-month growth metrics.

In [ ]:
mom = monthly[['sum']].rename(columns={'sum': 'revenue'})
mom['prev'] = mom['revenue'].shift(1)
mom['mom_pct'] = ((mom['revenue'] - mom['prev']) / mom['prev'] * 100).round(1)
print(mom.head())

### Recap

A datetime index enables time slicing; `resample` rolls rows to a coarser grain
(`D`/`W`/`ME`); `rolling` computes moving averages; `shift` powers
period-over-period change. Next: columnar file formats and Parquet.